In [ ]:
!pip install -U langchain langchain-openai
import openai
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, AIMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

os.environ['OPENAI_API_KEY'] = ""

openai.api_key = os.getenv('OPENAI_API_KEY')

llm = ChatOpenAI(openai_api_key=openai.api_key, model_name="gpt-3.5-turbo")

In [6]:
class My_List(BaseModel):
    name: str = Field(description="")
    list_items: list = Field(description="")

def list_creator(topic):
    My_List.name = f"Name of the list of the {topic}"
    My_List.list_items = f"A list of the {topic}"

    parser = PydanticOutputParser(pydantic_object=My_List)

    human_text = "{instruction}\n{format_instructions}"
    human_prompt = HumanMessagePromptTemplate.from_template(human_text)

    chat_prompt = ChatPromptTemplate.from_messages([human_prompt])

    prompt = chat_prompt.format_prompt(
        instruction=f"Create a list for {topic}",
        format_instructions=parser.get_format_instructions()
    ).to_messages()

    reply = llm.invoke(prompt)

    return parser.parse(reply.content)


def tweet_creator(topic, tone):
    system_template = "You are a social media expert."
    system_message = SystemMessagePromptTemplate.from_template(system_template)

    human_template = "Write a tweet about {topic}. The tone should be {tone}. The tweet should be no longer than 240 characters."
    human_message = HumanMessagePromptTemplate.from_template(human_template)

    chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

    prompt = chat_prompt.format_prompt(topic=topic, tone=tone).to_messages()
    reply=llm.invoke(prompt)

    return reply.content

In [9]:
topic = input("Please enter a topic for a list: ")
result = list_creator(topic)
print(result)
for i, item in enumerate(result.list_items, start=1):
    print(f"{i}. {item}")

topic_number = int(input("Enter the topic number to make a tweet about it."))
tone = input("Enter the tone for the tweet.")
tweet = tweet_creator(result.list_items[topic_number - 1], tone)

print("Tweet:")
print(f"The tweet: {tweet}")

Please enter a topic for a list: visiting zoo'
name='Visiting Zoo' list_items=['Check the schedule and plan the visit accordingly', 'Wear comfortable clothing and shoes', 'Bring water and snacks', 'Observe and learn about different animal species', 'Take photos and videos of the animals', "Follow the zoo's rules and guidelines", 'Enjoy the day with family and friends']
1. Check the schedule and plan the visit accordingly
2. Wear comfortable clothing and shoes
3. Bring water and snacks
4. Observe and learn about different animal species
5. Take photos and videos of the animals
6. Follow the zoo's rules and guidelines
7. Enjoy the day with family and friends
Enter the topic number to make a tweet about it.4
Enter the tone for the tweet.happy
Tweet:
The tweet: "Exploring the wonderful world of animals is such a joy! 🌿🐾 Observing their unique behaviors and characteristics is an endless source of fascination. Let's continue to learn and appreciate the amazing diversity of animal species aro